# 05. HMM follower: forward-распределение по позициям партитуры

Алгоритм по Rabiner 1989 [4]. Скрытые состояния `i ∈ {0, …, N−1}` — индексы нот партитуры. На каждом MIDI-событии выполняется один шаг forward:

$$\alpha_{t+1}(i) = \Big[\sum_j \alpha_t(j) \cdot a_{j,i}\Big] \cdot b_i(o_{t+1})$$

**Эмиссия** $b_i(o)$ — гауссиана по pitch:

$$b_i(\text{pitch}) = \frac{1}{\sigma\sqrt{2\pi}}\exp\Big(-\tfrac{1}{2}\big(\tfrac{\text{pitch}-\text{pitch}_i}{\sigma}\big)^2\Big)$$

**Переходы** $a_{j,i}$ — три варианта: остаться в `j` (stay), сдвинуться на `j+1` (advance), перепрыгнуть на `j+2` (skip). Веса переходов **зависят от времени, проведённого в текущем state** — это уже немного HSMM:

```python
if elapsed > 1.5 * nominal_duration:   advance_p = 0.95
elif elapsed >= nominal_duration:      advance_p = 0.80
else:                                   advance_p = 0.60
```

В этом ноутбуке мы:

1. Прогоняем `ScoreFollowerHMM` на `ideal` и сохраняем α на каждом шаге.
2. Рисуем тепловую карту α (timestep × state).
3. Смотрим как эмиссия зависит от наблюдаемого pitch.
4. Сравниваем поведение на rubato и noisy.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt

from musictech.core import ScoreFollowerHMM
from dataset_viewer import load_performance

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
DATA = PROJECT_ROOT / "generated_dataset"

## 1. Прогон с записью α-распределения

In [ ]:
def run_hmm_recording_alpha(score_path: Path, perf_path: Path, sigma: float = 2.0):
    follower = ScoreFollowerHMM(str(score_path), sigma=sigma)
    performance = load_performance(perf_path)
    alphas = []           # каждая строка: α после i-го события, shape (N,)
    predicted = []
    for event in performance:
        idx = follower.process_event(event)
        alphas.append(follower.alpha.copy())
        predicted.append(idx)
    return follower, performance, np.array(alphas), predicted

follower, performance, alphas, predicted = run_hmm_recording_alpha(
    DATA / "ideal.json", DATA / "ideal.mid"
)
print("alpha matrix shape (events × states):", alphas.shape)
print("predicted:", predicted)
print("expected :", list(range(follower.N)))

## 2. Тепловая карта α: куда смотрит трекер

Каждая строка — α после очередного события. Светлые квадраты — состояния с высокой вероятностью.

In [ ]:
def plot_alpha_heatmap(alphas: np.ndarray, predicted: list[int], title: str):
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(alphas, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
    ax.plot(predicted, range(len(predicted)), color="white", marker="o", linestyle="", label="argmax")
    ax.set_xlabel("score state index")
    ax.set_ylabel("event #")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="α(state)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

plot_alpha_heatmap(alphas, predicted, "HMM α on 'ideal'")

## 3. Эмиссионная функция

Зачем гауссиана а не дельта-функция? Чтобы устойчиво ловить «соседние клавиши» — пианист может сыграть `C` вместо `D` (ошибка на 2 полутона), а трекеру должно быть всё равно. Параметр `sigma` (в полутонах) регулирует терпимость.

In [ ]:
score_pitches = follower.pitches    # реальный массив score-pitch-ей
pitch_range = np.arange(55, 78, 0.1)
fig, ax = plt.subplots()
for sigma in [0.5, 1.0, 2.0, 4.0]:
    f = ScoreFollowerHMM(str(DATA / "ideal.json"), sigma=sigma)
    emissions = np.array([f._emission_probabilities(p) for p in pitch_range])  # (R, N)
    ax.plot(pitch_range, emissions[:, 0], label=f"σ = {sigma} (state 0, pitch={int(score_pitches[0])})")
ax.set_xlabel("observed pitch")
ax.set_ylabel("b_0(pitch)")
ax.set_title("Emission probability for state 0 (pitch=60) at different σ")
ax.legend()
plt.show()

## 4. Поведение на rubato

In [ ]:
follower, performance, alphas, predicted = run_hmm_recording_alpha(
    DATA / "rubato.json", DATA / "rubato.mid"
)
plot_alpha_heatmap(alphas, predicted, "HMM α on 'rubato'")

## 5. Поведение на шуме

На `noisy.mid` пианист (синтетически) делает опечатки и пропуски. HMM не имеет явной модели «лишняя нота», поэтому реагирует медленно и иногда уходит в неверное состояние. Это мотивация для HSMM (ноутбук 06) и тем более для `HybridScoreFollower` (ноутбук 07).

In [ ]:
follower, performance, alphas, predicted = run_hmm_recording_alpha(
    DATA / "noisy.json", DATA / "noisy.mid"
)
print("perf pitches  :", [int(e["pitch"]) for e in performance])
print("predicted idx :", predicted)
plot_alpha_heatmap(alphas, predicted, "HMM α on 'noisy'")

## 6. Что важно для тезисов

- **`α_t` — это и есть состояние, которое RL-агенту нужно как часть `s_t`** (см. формулу из тезисов). Полный вектор длины N передавать в политику нельзя — он зависит от пьесы. Поэтому в `musictech.core.dto.AlphaSummary` мы собираем 7 чисел: `max`, `entropy`, `argmax/N`, `top-3 mass` и три top-индекса. См. ноутбук 09.
- **Текущая модель — это HMM + наивная зависимость переходов от времени**. Полноценный HSMM по Cont 2010 — следующий шаг (трек 2A из `analysis.md`).